# 🎬 AutoShorts — Google Colab Pipeline (GPU)

**AutoShorts** turns long-form video/audio into viral **9:16 vertical Shorts** with AI-powered moment ranking.
This notebook runs the **full pipeline on Google Colab's free NVIDIA GPU** — local Whisper transcription is GPU-accelerated (no cloud transcription API needed), and clip rendering uses GPU NVENC encoding when available.

- 📦 Repo: [mausamrijall/autoshorts](https://github.com/mausamrijall/autoshorts)
- 🔁 Pipeline: `ingest → extract audio → transcribe (GPU Whisper) → LLM moment detection → render vertical clips`
- 🐍 `colab/autoshorts_pipeline.py` is a faithful Python port of the desktop app's Rust backend (`media.rs`, `transcription.rs`, `llm.rs`).

> 💡 **Recommended setup:** use the **DeepSeek** provider for moment detection (cheap, fast, accurate) and local GPU Whisper for transcription. You can also run fully offline with local **Ollama** models (optional cell below).


In [ ]:
# ═══════════════════════════ CONFIG ═══════════════════════════
# Edit this cell before running (or set Colab secrets for API keys).

# ── Input ────────────────────────────────────────────────────────
# Leave empty to upload a file from your computer instead.
YOUTUBE_URL = ""

# ── Transcription ────────────────────────────────────────────────
# "local"    = Whisper on the Colab GPU (default — NO API key needed)
# "deepgram" = Deepgram cloud transcription (needs DEEPGRAM_API_KEY)
TRANSCRIPTION_MODE = "local"
WHISPER_MODEL = "base"          # tiny|base|small|medium|large-v3 (bigger = better + slower)

# ── Moment detection (LLM) ────────────────────────────────────────
# deepseek (default, cheapest) | claude | gemini | openai | groq | openrouter
# | openai_compat (any OpenAI-compatible proxy/gateway) | ollama (offline)
LLM_PROVIDER = "deepseek"

# Required when LLM_PROVIDER == "openai_compat": the API root that serves
# /chat/completions — e.g. https://openrouter.ai/api/v1 or your own gateway.
OPENAI_COMPAT_BASE_URL = ""
OPENAI_COMPAT_MODEL = ""

# Optional model overrides (empty string = provider default)
DEEPSEEK_MODEL = ""
ANTHROPIC_MODEL = ""
GEMINI_MODEL = ""
OPENAI_MODEL = ""
GROQ_MODEL = ""
OPENROUTER_MODEL = ""
OLLAMA_MODEL = "qwen2.5:7b"     # used when LLM_PROVIDER == "ollama"

# ── Rendering ─────────────────────────────────────────────────────
NUM_CLIPS = 5                    # how many clips to render (max 10)
CAPTION_STYLE = "modern-box"     # modern-box | classic-outline | minimal-shadow | vibrant-cyan | vibrant-yellow-box | vibrant-green
RENDER_WITH_CAPTIONS = True


In [ ]:
# ── Environment & GPU check ───────────────────────────────────────
import os, sys, shutil, platform, subprocess

print("Python:", platform.python_version(), "| machine:", platform.machine())
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "[WARN] no NVIDIA GPU detected — Whisper will fall back to CPU"

# System dependency (ffmpeg/ffprobe for audio extraction + clip rendering)
!apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("ffmpeg:", shutil.which("ffmpeg"), "| ffprobe:", shutil.which("ffprobe"))

# Python deps. PyTorch+CUDA is already pre-installed in Colab, so openai-whisper
# will use the existing CUDA-enabled torch.
!pip install -q yt-dlp openai-whisper requests
print("deps installed")


In [ ]:
# ── Clone AutoShorts from GitHub ───────────────────────────────────
REPO_URL = "https://github.com/mausamrijall/autoshorts.git"
REPO_DIR = "/content/autoshorts"
BRANCH = "main"

def _sh(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.strip() or f"command failed: {cmd}")
    return r

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    _sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])
else:
    _sh(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH])

# Make the pipeline module importable
sys.path.insert(0, os.path.join(REPO_DIR, "colab"))
print("cloned to", REPO_DIR)


In [ ]:
# ── API keys (optional depending on provider) ──────────────────────
import getpass
try:
    from google.colab import userdata
except ImportError:
    userdata = None

def _secret(name: str):
    if userdata is not None:
        try:
            return userdata.get(name)
        except Exception:
            return None
    return None

API_KEYS = {
    "DEEPGRAM_API_KEY": _secret("DEEPGRAM_API_KEY"),
    "DEEPSEEK_API_KEY": _secret("DEEPSEEK_API_KEY"),
    "ANTHROPIC_API_KEY": _secret("ANTHROPIC_API_KEY"),
    "GEMINI_API_KEY": _secret("GEMINI_API_KEY"),
    "OPENAI_API_KEY": _secret("OPENAI_API_KEY"),
    "GROQ_API_KEY": _secret("GROQ_API_KEY"),
    "OPENROUTER_API_KEY": _secret("OPENROUTER_API_KEY"),
    "OPENAI_COMPAT_API_KEY": _secret("OPENAI_COMPAT_API_KEY"),
}

_KEY_FOR_PROVIDER = {
    "deepgram": "DEEPGRAM_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "claude": "ANTHROPIC_API_KEY",
    "gemini": "GEMINI_API_KEY",
    "openai": "OPENAI_API_KEY",
    "groq": "GROQ_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
    "openai_compat": "OPENAI_COMPAT_API_KEY",
}

needed = set()
if TRANSCRIPTION_MODE == "deepgram":
    needed.add(_KEY_FOR_PROVIDER["deepgram"])
if LLM_PROVIDER != "ollama":
    needed.add(_KEY_FOR_PROVIDER[LLM_PROVIDER])

for key in needed:
    if not API_KEYS.get(key):
        API_KEYS[key] = getpass.getpass(f"Paste {key}: ")

for key in sorted(needed):
    print(f"{key}: {'✓ set' if API_KEYS.get(key) else '✗ missing'}")


In [ ]:
# ── Input media: YouTube URL OR file upload ─────────────────────────
import pathlib, shutil, time, json

WORK = pathlib.Path("/content/autoshorts_work")
SOURCE_DIR = WORK / "source"
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

if YOUTUBE_URL.strip():
    print("Downloading YouTube video …")
    _sh([
        "yt-dlp", "-f", "bv*+ba/b", "--merge-output-format", "mp4",
        "-o", f"{SOURCE_DIR}/input.%(ext)s", YOUTUBE_URL,
    ])
else:
    from google.colab import files
    uploaded = files.upload()          # pick your video/audio file
    for name, data in uploaded.items():
        (SOURCE_DIR / name).write_bytes(data)

media_files = sorted(SOURCE_DIR.iterdir())
assert media_files, "No media file found. Upload one above or set YOUTUBE_URL."
SOURCE_PATH = str(media_files[0])
print("Source media:", SOURCE_PATH)

import autoshorts_pipeline as ap


In [ ]:
# ── (Optional) Local Ollama with GPU for fully-offline detection ──
# Only runs when LLM_PROVIDER == "ollama". Uses the Colab GPU via CUDA.
if LLM_PROVIDER == "ollama":
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(8)
    _sh(["ollama", "pull", OLLAMA_MODEL])


In [ ]:
# ── Step 1–2: probe media + extract audio + transcribe (GPU) ───────
probe = ap.probe_media(SOURCE_PATH)
print(f"input: {probe['width']}x{probe['height']}, duration={probe['duration_sec']:.1f}s, video={probe['has_video']}")

audio_path = ap.extract_audio(SOURCE_PATH, WORK / "project")
print("audio:", audio_path)

if TRANSCRIPTION_MODE == "deepgram":
    transcript = ap.transcribe_deepgram(audio_path, API_KEYS["DEEPGRAM_API_KEY"])
    engine = "Deepgram cloud"
else:
    import torch
    transcript = ap.transcribe_local(audio_path, model_size=WHISPER_MODEL)
    device = "CUDA" if torch.cuda.is_available() else "CPU"
    engine = f"Whisper/{WHISPER_MODEL} on {device}"

print(f"transcribed with {engine}: {len(transcript['words'])} words, {len(transcript['segments'])} segments")


In [ ]:
# ── Step 3: LLM viral-moment detection ────────────────────────────
if LLM_PROVIDER == "ollama":
    candidates = ap.detect_candidates_with_local_llm(transcript, OLLAMA_MODEL)
elif LLM_PROVIDER == "openai_compat":
    assert OPENAI_COMPAT_BASE_URL.strip(), "Set OPENAI_COMPAT_BASE_URL in CONFIG."
    candidates = ap.detect_candidates_with_openai_compat(
        transcript,
        API_KEYS["OPENAI_COMPAT_API_KEY"],
        OPENAI_COMPAT_BASE_URL,
        OPENAI_COMPAT_MODEL or None,
    )
else:
    fn = getattr(ap, f"detect_candidates_with_{LLM_PROVIDER}")
    key = API_KEYS[_KEY_FOR_PROVIDER[LLM_PROVIDER]]
    model_override = {
        "deepseek": DEEPSEEK_MODEL,
        "claude": ANTHROPIC_MODEL,
        "gemini": GEMINI_MODEL,
        "openai": OPENAI_MODEL,
        "groq": GROQ_MODEL,
        "openrouter": OPENROUTER_MODEL,
    }[LLM_PROVIDER] or None
    candidates = fn(transcript, key, model_override)

print(f"Detected {len(candidates)} clip candidates via {LLM_PROVIDER}")

import pandas as pd
df = pd.DataFrame(candidates)
df["duration_s"] = (df["end"] - df["start"]).round(1)
df = df[["start", "end", "duration_s", "score", "hook", "rationale"]]
df.columns = ["Start (s)", "End (s)", "Duration (s)", "Score", "Hook", "Rationale"]
from IPython.display import display
display(df)


In [ ]:
# ── Step 4: render top-N vertical clips + preview + download ───────
from IPython.display import Video
CLIPS_DIR = WORK / "clips"
CLIPS_DIR.mkdir(exist_ok=True)
crop_w = ap._cropped_width(probe)

for rank, cand in enumerate(candidates[:NUM_CLIPS], start=1):
    dst = CLIPS_DIR / f"clip-{rank:02d}_flat.mp4"
    drawtext = None
    if RENDER_WITH_CAPTIONS:
        drawtext = ap.build_drawtext_filters(
            transcript["words"], cand["start"], cand["end"], crop_w, CAPTION_STYLE
        )
    try:
        ap.render_flat_clip(SOURCE_PATH, cand["start"], cand["end"], dst, drawtext, gpu_encode=True)
    except Exception as exc:
        print(f"[warn] captions failed ({exc}); rendering clean clip")
        ap.render_flat_clip(SOURCE_PATH, cand["start"], cand["end"], dst, gpu_encode=True)

    print(f"rendered {dst.name}  [{cand['start']:.1f}s – {cand['end']:.1f}s]  score={cand['score']:.2f}")
    display(Video(str(dst), embed=True, width=270))

zip_path = "/content/autoshorts_clips.zip"
shutil.make_archive(zip_path[:-4], "zip", CLIPS_DIR)
print("\nAll clips zipped →", zip_path)
from google.colab import files
files.download(zip_path)


## 📋 Notes & troubleshooting

- **GPU**: Colab assigns a GPU (e.g. T4) via *Runtime → Change runtime type → Hardware accelerator → GPU*. Without a GPU, Whisper falls back to CPU (slower but works).
- **GPU rendering**: clips are encoded on the GPU via **NVENC** (`h264_nvenc`) with automatic CPU `libx264` fallback.
- **API keys**: For cloud providers, add them as Colab *Secrets* (the 🔑 key icon in the left sidebar) or paste them when prompted. Recommended: [DeepSeek](https://platform.deepseek.com) for moment detection.
- **Custom provider**: set `LLM_PROVIDER = "openai_compat"` and `OPENAI_COMPAT_BASE_URL` to use any OpenAI-compatible proxy/gateway (e.g. OpenRouter, LiteLLM, your own backend).
- **Fully offline**: set `LLM_PROVIDER = "ollama"` — the optional Ollama cell installs a local model on the GPU, so no API keys are needed at all.
- **Whisper model sizes**: `tiny`/`base` are fast; `small`/`medium`/`large-v3` are more accurate but slower. If you change `WHISPER_MODEL` mid-session, restart the runtime (the model is cached in memory).
- **Outputs** live in `/content/autoshorts_work/` (clips + transcript JSON) until the runtime ends — download the ZIP before closing.
- **Captions**: if ffmpeg's `drawtext` fails (rare font issues), clips render clean without captions.
- **9:16 crop**: landscape input is center-cropped to a vertical strip, exactly like the desktop app.
